# RetinaNet 모델

### git 연결

In [ ]:
# 1. 내 포크 clone
!git clone https://github.com/beomsookim1020/pill_detection_project.git

# 2. 프로젝트 폴더로 이동
%cd pill_detection_project

# 3. requirements 설치
!pip install --upgrade pip
!pip install --index-url https://download.pytorch.org/whl/cu118 torch==2.7.1+cu118 torchvision==0.22.1+cu118 torchaudio==2.7.1+cu118
!pip install -r requirements.txt --no-deps


Cloning into 'pill_detection_project'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 135 (delta 0), reused 0 (delta 0), pack-reused 132 (from 1)
Receiving objects: 100% (135/135), 464.52 KiB | 1.93 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/pill_detection_project/pill_detection_project
Looking in indexes: https://download.pytorch.org/whl/cu118


In [ ]:
import sys
import os
sys.path.append("/content/pill_detection_project")

from src.evaluation import (
    evaluate_all,
    init_history,
    update_history,
    save_history,
    load_history,
    plot_training_history,
    plot_compare_histories,
    convert_yolo_results,
    convert_torchvision_outputs,
)


### 구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import unicodedata  # 0번 섹션에 추가 필요

# 📌 경로 설정 (제공해주신 경로 반영)
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

# 데이터 가져오기

In [ ]:
############################################################
# 0. 라이브러리 임포트 & 경로 설정
############################################################
import os
import json
import pandas as pd
from PIL import Image
import unicodedata  # 0번 섹션에 추가 필요
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# 📌 경로 설정 (제공해주신 경로 반영)
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

# 경로는 환경에 맞게 수정
# train_images, test_images
extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/") # 압축을 풀 폴더

TRAIN_JSON_PATH = os.path.join(extract_path, "merged_annotations_train_final.json")
TEST_JSON_PATH = os.path.join(extract_path, "merged_annotations_test_final.json")
TRAIN_IMG_DIR = os.path.join(extract_path, "train_images")
TEST_IMG_DIR  = os.path.join(extract_path, "test_images")

# merged_annotation json 경로
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

############################################################
# 1. 병합된 JSON 파일을 읽어서 DataFrame으로 만들기
############################################################

def build_df_from_merged_json(json_path, img_dir):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 1) 이미지 정보 매핑 (id -> file_name)
    id_to_fname = {img["id"]: img["file_name"] for img in data["images"]}

    records = []
    # 2) 어노테이션 순회
    for ann in data["annotations"]:
        img_id_coco = ann["image_id"]
        file_name = id_to_fname.get(img_id_coco)
        
        if file_name is None: continue
        
        img_path = os.path.join(img_dir, file_name)
        
        # 실제 이미지 파일이 있는지 확인 (선택 사항이지만 안전함)
        if not os.path.exists(img_path):
            continue

        x, y, w, h = ann["bbox"]
        
        records.append({
            "image_path": img_path,
            "image_id": os.path.splitext(file_name)[0], # 파일명을 ID로 사용
            "category_id": int(ann["category_id"]),
            "bbox_x": float(x),
            "bbox_y": float(y),
            "bbox_w": float(w),
            "bbox_h": float(h),
        })

    return pd.DataFrame(records)

# 실행
df = build_df_from_merged_json(TRAIN_JSON_PATH, TRAIN_IMG_DIR)
print(f"✅ 학습 데이터 로드 완료: {len(df)} 개의 객체 탐지됨")

✅ 학습 데이터 로드 완료: 4526 개의 객체 탐지됨


In [ ]:
############################################################
# 2. category_id 매핑 (겉으로는 안 바꾸고, 모델 내부에서만 사용)
############################################################

# 원본 category_id 집합
unique_cats = sorted(df["category_id"].unique())
print("고유 category_id 개수:", len(unique_cats))

# 내부용: 모델에 넣을 label (1 ~ num_classes-1), 0은 background
orig2model = {cid: i + 1 for i, cid in enumerate(unique_cats)}   # 원본 → 모델용
model2orig = {v: k for k, v in orig2model.items()}               # 모델용 → 원본

num_classes = len(unique_cats) + 1  # background 포함
print("num_classes (background 포함):", num_classes)

고유 category_id 개수: 73
num_classes (background 포함): 74


In [ ]:
############################################################
# 3. Dataset 정의
############################################################

class OralDrugDataset(Dataset):
    """
    Faster R-CNN용 Dataset
    - __getitem__은 (image, target) 반환
    - image: [C,H,W] float tensor
    - target: dict(boxes, labels, image_id, ...)
    """
    def __init__(self, df, orig2model, transforms=None):
        self.df = df.reset_index(drop=True)
        self.orig2model = orig2model
        self.transforms = transforms

        # 이미지 단위로 그룹을 만들기 위해 고유 image_id 리스트를 유지
        self.image_ids = self.df["image_id"].unique().tolist()

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        # 1) image_id 선택
        image_id = self.image_ids[idx]

        # 2) 해당 image_id에 해당하는 모든 row (여러 객체) 가져오기
        df_img = self.df[self.df["image_id"] == image_id]

        # 3) 이미지 로드
        img_path = df_img["image_path"].iloc[0]
        image = Image.open(img_path).convert("RGB")

        # 4) bbox / labels 구성
        boxes = []
        labels = []

        for _, row in df_img.iterrows():
            x = row["bbox_x"]
            y = row["bbox_y"]
            w = row["bbox_w"]
            h = row["bbox_h"]

            # [x1, y1, x2, y2] 형식으로 변환
            x1 = x
            y1 = y
            x2 = x + w
            y2 = y + h
            boxes.append([x1, y1, x2, y2])

            # 원본 category_id → 모델용 label로 변환
            orig_cat = int(row["category_id"])
            model_cat = self.orig2model[orig_cat]
            labels.append(model_cat)

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            # image_id는 loss에는 크게 영향 없음, 그냥 인덱스 정도로 넣어도 무방
            "image_id": torch.tensor([idx]),
        }

        if self.transforms is not None:
            image = self.transforms(image)

        return image, target


############################################################
# 4. Transform, Dataset, DataLoader 구성
############################################################

train_transforms = T.Compose([
    # 필요하다면 여기서 RandomHorizontalFlip 등 augmentation 추가
    T.ToTensor(),  # PIL 이미지를 [0,1] float tensor로 변환
])

dataset = OralDrugDataset(df, orig2model=orig2model, transforms=train_transforms)

# 간단히 train/val split (예: 90% train, 10% val)
indices = torch.randperm(len(dataset)).tolist()
split = int(0.9 * len(indices))
train_indices = indices[:split]
val_indices   = indices[split:]

train_dataset = torch.utils.data.Subset(dataset, train_indices)
val_dataset   = torch.utils.data.Subset(dataset, val_indices)

def collate_fn(batch):
    # detection 모델용 collate_fn: 리스트의 튜플을 튜플의 리스트로
    return tuple(zip(*batch))

train_loader = DataLoader(
    train_dataset,
    batch_size=2,              # GPU 메모리에 맞게 조정
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn,
)

print("train steps:", len(train_loader), "val steps:", len(val_loader))



train steps: 670 val steps: 75


In [ ]:
############################################################
# 11. RetinaNet 모델 정의 + 학습
############################################################

from torchvision.models.detection import retinanet_resnet50_fpn
from torchvision.models.detection.retinanet import RetinaNetClassificationHead

def build_retinanet_model(num_classes):
    model = retinanet_resnet50_fpn(weights="DEFAULT")
    cls_head = model.head.classification_head

    # torchvision 버전에 따라 conv[0]이 Conv2dNormActivation일 수 있으므로
    # 내부 첫 Conv2d를 찾아 채널 수를 읽는다.
    first_conv = next(
        module for module in cls_head.conv.modules() if isinstance(module, nn.Conv2d)
    )
    in_channels = first_conv.in_channels
    num_anchors = cls_head.num_anchors

    model.head.classification_head = RetinaNetClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=num_classes,
    )
    return model


# 사전학습된 RetinaNet 모델 로드
model = build_retinanet_model(num_classes)
model.to(DEVICE)

# 옵티마이저 / 스케줄러 설정
retina_optimizer = optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=1e-4,
)
retina_scheduler = optim.lr_scheduler.StepLR(retina_optimizer, step_size=3, gamma=0.1)


############################################################
# 12. RetinaNet 학습 루프
############################################################

num_epochs = 1

for epoch in range(1, num_epochs+1):
    ##########################
    # 1) Train
    ##########################
    model.train()
    train_loss_sum = 0.0

    for images, targets in train_loader:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        retina_optimizer.zero_grad()
        losses.backward()
        retina_optimizer.step()

        train_loss_sum += losses.item()

    epoch_train_loss = train_loss_sum / max(1, len(train_loader))

    ##########################
    # 2) Validation (loss만 측정)
    ##########################
    model.train()  # detection 모델은 train() 상태에서 loss를 반환
    val_loss_sum = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            val_loss_sum += losses.item()

    epoch_val_loss = val_loss_sum / max(1, len(val_loader))

    print(
        f"[RetinaNet Epoch {epoch+1}/{num_epochs}] "
        f"train_loss: {epoch_train_loss:.4f} | "
        f"val_loss: {epoch_val_loss:.4f}"
    )

    retina_scheduler.step()

# 학습된 모델 저장
torch.save(model.state_dict(), "retinanet_oral_drug.pth")
print("RetinaNet 모델 저장 완료")



[RetinaNet Epoch 2/1] train_loss: 0.8656 | val_loss: 0.5867
RetinaNet 모델 저장 완료


In [ ]:
history = init_history()

for epoch in range(1, num_epochs + 1):
    # 1. train
    train_loss = epoch_train_loss

    # 2. val
    val_loss = epoch_val_loss

    # 3. 검증셋 예측
    all_outputs = []
    all_image_ids = []

    model.eval()
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            outputs = model(images)

            batch_image_ids = [t["image_id"].item() for t in targets]

            all_outputs.extend(outputs)
            all_image_ids.extend(batch_image_ids)

    val_predictions = convert_torchvision_outputs(all_outputs, all_image_ids)

    # 4. 평가
    metrics = evaluate_all(
        gt_json_path=VAL_JSON_PATH,
        predictions=val_predictions,
        conf_threshold=0.25,
        pr_iou_threshold=0.5,
        temp_json_path=f"retinanet_temp_eval_epoch_{epoch}.json"
    )

    # 5. 기록
    update_history(
        history,
        epoch=epoch,
        train_loss=train_loss,
        val_loss=val_loss,
        metrics=metrics
    )

save_history(history, "history_retinanet.json")
plot_training_history(history, title_prefix="RetinaNet")

NameError: name 'VAL_JSON_PATH' is not defined